# Notebook 05: Analysis & Paper Tables

Produces all statistics, tables, and figures for the paper.

**Inputs:** `judge_scores.json`, `embed_distances.json`  
**Outputs:** `results/final_table.csv`, figures

In [ ]:
!pip install -q scipy pandas matplotlib seaborn

In [ ]:
import json, os
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

BASE     = Path(".")
DATA_DIR = BASE / "../data"
FIG_DIR  = BASE / "../results/figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

judge_path = DATA_DIR / "judge_scores.json"
embed_path = DATA_DIR / "embed_distances.json"

with open(judge_path) as f: judge_data = json.load(f)
with open(embed_path)  as f: embed_data = json.load(f)

jdf = pd.DataFrame(judge_data)
edf = pd.DataFrame(embed_data)

# valid stances only (>0); mistral PA excluded (n=1, not meaningful)
jdf = jdf[jdf["stance"] > 0].copy()
jdf = jdf[~((jdf["model"] == "mistral_7b") & (jdf["lang"] == "pa"))]
print(f"Judge records after filtering: {len(jdf)}")
print(f"Embed records: {len(edf)}")


In [ ]:
# ── TABLE 1: Script Fidelity ──────────────────────────────────────────────────
# How many models can actually write Gurmukhi / Devanagari?

fidelity_rows = []
for model in jdf["model"].unique():
    for lang in ["hi", "pa"]:
        sub = jdf[(jdf["model"] == model) & (jdf["lang"] == lang)]
        pass_rate = sub["script_ok"].mean() * 100
        fidelity_rows.append({"Model": model, "Language": lang, "Script Fidelity %": round(pass_rate, 1)})

fidelity_df = pd.DataFrame(fidelity_rows).pivot(index="Model", columns="Language", values="Script Fidelity %")
print("\nTABLE 1: Script Fidelity")
print(fidelity_df.to_string())

In [ ]:
# ── TABLE 2: Cross-lingual Value Drift (judge scores) ────────────────────────
# For each model × dimension: mean stance in EN vs HI vs PA
# Drift = |stance_EN - stance_PA| and |stance_EN - stance_HI|

drift_rows = []
models = jdf["model"].unique()
dims   = jdf["dimension"].unique()

for model in models:
    for dim in dims:
        sub = jdf[(jdf["model"] == model) & (jdf["dimension"] == dim)]
        mean_en = sub[sub["lang"] == "en"]["stance"].mean()
        mean_hi = sub[sub["lang"] == "hi"]["stance"].mean()
        mean_pa = sub[sub["lang"] == "pa"]["stance"].mean()
        drift_rows.append({
            "model":       model,
            "dimension":   dim,
            "stance_EN":   round(mean_en, 2),
            "stance_HI":   round(mean_hi, 2),
            "stance_PA":   round(mean_pa, 2),
            "drift_EN_HI": round(abs(mean_en - mean_hi), 2),
            "drift_EN_PA": round(abs(mean_en - mean_pa), 2),
        })

drift_df = pd.DataFrame(drift_rows)
print("\nTABLE 2: Cross-lingual Value Drift (judge scores)")
print(drift_df.to_string(index=False))

In [ ]:

# ── STATISTICAL TESTS (H1, H2) ────────────────────────────────────────────────

# Exclude models with script fidelity failure from H1/H2:
#   phi35_mini     — 0% HI/PA fidelity (can't generate SA scripts)
#   deepseek_r1_7b — reasoning model responds in English monologue for HI/PA
SCRIPT_FAIL_MODELS = {"phi35_mini", "deepseek_r1_7b"}
SCRIPT_CAPABLE = [m for m in jdf["model"].unique() if m not in SCRIPT_FAIL_MODELS]
jdf_stats = jdf[jdf["model"].isin(SCRIPT_CAPABLE)]
print(f"Models excluded from H1/H2 (script failure): {SCRIPT_FAIL_MODELS}")
print(f"Models included in H1/H2 ({len(SCRIPT_CAPABLE)}): {SCRIPT_CAPABLE}")

print("\n── H1: Is drift(EN→PA) significantly > 0 across script-capable models? ──")

en_stances, pa_stances, hi_stances = [], [], []
for _, row in jdf_stats[jdf_stats["lang"] == "en"].iterrows():
    pa_row = jdf_stats[
        (jdf_stats["model"] == row["model"]) &
        (jdf_stats["scenario_id"] == row["scenario_id"]) &
        (jdf_stats["religion"] == row["religion"]) &
        (jdf_stats["lang"] == "pa")
    ]
    hi_row = jdf_stats[
        (jdf_stats["model"] == row["model"]) &
        (jdf_stats["scenario_id"] == row["scenario_id"]) &
        (jdf_stats["religion"] == row["religion"]) &
        (jdf_stats["lang"] == "hi")
    ]
    if len(pa_row) and len(hi_row):
        en_stances.append(row["stance"])
        pa_stances.append(pa_row.iloc[0]["stance"])
        hi_stances.append(hi_row.iloc[0]["stance"])

en_arr = np.array(en_stances)
pa_arr = np.array(pa_stances)
hi_arr = np.array(hi_stances)

stat_en_pa, p_en_pa = stats.wilcoxon(en_arr, pa_arr)
stat_en_hi, p_en_hi = stats.wilcoxon(en_arr, hi_arr)

drift_en_pa = np.abs(en_arr - pa_arr)
drift_en_hi = np.abs(en_arr - hi_arr)

def cohens_d(a, b):
    return (np.mean(a) - np.mean(b)) / np.sqrt((np.std(a)**2 + np.std(b)**2) / 2)

d_h1 = cohens_d(pa_arr, en_arr)
print(f"H1 EN vs PA: W={stat_en_pa:.1f}, p={p_en_pa:.4f}, mean_drift={drift_en_pa.mean():.2f}, Cohen's d={d_h1:.3f}")
d_h2_raw = cohens_d(hi_arr, en_arr)
print(f"H1 EN vs HI: W={stat_en_hi:.1f}, p={p_en_hi:.4f}, mean_drift={drift_en_hi.mean():.2f}, Cohen's d={d_h2_raw:.3f}")

print(f"\n── H2: Is drift(EN→PA) > drift(EN→HI)? ──")
stat_h2, p_h2 = stats.wilcoxon(drift_en_pa, drift_en_hi)
d_h2 = cohens_d(drift_en_pa, drift_en_hi)
print(f"H2 Wilcoxon: W={stat_h2:.1f}, p={p_h2:.4f}, Cohen's d={d_h2:.3f}")
print(f"Mean drift EN→PA: {drift_en_pa.mean():.3f} vs EN→HI: {drift_en_hi.mean():.3f}")


In [ ]:

# ── H3: Does Aya Expanse 8B (multilingual specialist) show less drift? ────────
# Uses jdf_stats (script-capable models only — excludes phi35_mini, deepseek_r1_7b)

print("\n── H3: Aya Expanse 8B vs other script-capable models drift(EN→PA) ──")
aya_drift, other_drift = [], []
for _, row in jdf_stats[jdf_stats["lang"] == "en"].iterrows():
    pa_row = jdf_stats[
        (jdf_stats["model"] == row["model"]) &
        (jdf_stats["scenario_id"] == row["scenario_id"]) &
        (jdf_stats["religion"] == row["religion"]) &
        (jdf_stats["lang"] == "pa")
    ]
    if len(pa_row):
        d = abs(row["stance"] - pa_row.iloc[0]["stance"])
        if row["model"] == "aya_expanse_8b":
            aya_drift.append(d)
        else:
            other_drift.append(d)

if len(aya_drift) > 0:
    stat_h3, p_h3 = stats.mannwhitneyu(aya_drift, other_drift, alternative="two-sided")
    print(f"Aya Expanse 8B mean drift: {np.mean(aya_drift):.3f}  (n={len(aya_drift)})")
    print(f"Other models mean drift:   {np.mean(other_drift):.3f}  (n={len(other_drift)})")
    print(f"MannWhitneyU: U={stat_h3:.1f}, p={p_h3:.4f}")
    if p_h3 < 0.05:
        if np.mean(aya_drift) > np.mean(other_drift):
            print("H3 NOT supported: Aya drifts significantly MORE than other models (counterintuitive finding).")
        else:
            print("H3 SUPPORTED: Aya drifts significantly less than other models.")
    else:
        print("H3 NOT supported: no significant difference in drift.")
else:
    print("aya_expanse_8b not found in dataset.")

# ── Script fidelity failure as a finding ─────────────────────────────────────
print("\n── Script fidelity failure models (excluded from H1-H3) ──")
for m in SCRIPT_FAIL_MODELS:
    sub = jdf[jdf["model"] == m]
    for lang in ["hi", "pa"]:
        rate = sub[sub["lang"] == lang]["script_ok"].mean() * 100
        print(f"  {m} | {lang}: {rate:.1f}% fidelity")


In [ ]:

import os
os.makedirs("/kaggle/working/results", exist_ok=True)

# ── FIGURE 1: Drift heatmap by model × dimension ─────────────────────────────
pivot = drift_df.pivot_table(
    values="drift_EN_PA", index="model", columns="dimension", aggfunc="mean"
)

n_models = len(pivot)
fig, ax = plt.subplots(figsize=(9, max(4, n_models * 0.65)))
sns.heatmap(pivot, annot=True, fmt=".2f", cmap="YlOrRd",
            linewidths=0.5, ax=ax, vmin=0, vmax=2)
ax.set_title("Cross-lingual Value Drift EN→Punjabi by Model & Dimension", fontsize=12)
ax.set_xlabel("Hofstede Dimension")
ax.set_ylabel("Model")
plt.tight_layout()
plt.savefig("/kaggle/working/results/fig1_drift_heatmap.pdf", bbox_inches="tight")
plt.show()
print("Figure 1 saved.")


In [ ]:
# ── FIGURE 2: EN vs HI vs PA stance per dimension (boxplot) ──────────────────

fig, axes = plt.subplots(1, 4, figsize=(14, 4), sharey=True)
dims = jdf["dimension"].unique()

for ax, dim in zip(axes, dims):
    sub = jdf[jdf["dimension"] == dim]
    data = [
        sub[sub["lang"] == "en"]["stance"].values,
        sub[sub["lang"] == "hi"]["stance"].values,
        sub[sub["lang"] == "pa"]["stance"].values,
    ]
    ax.boxplot(data, labels=["EN", "HI", "PA"])
    ax.set_title(dim, fontsize=9)
    ax.set_ylim(0.5, 5.5)
    ax.set_ylabel("Stance (1=Western, 5=South Asian)" if ax == axes[0] else "")

plt.suptitle("Value Stance Distribution by Language and Dimension", y=1.02)
plt.tight_layout()
plt.savefig("/kaggle/working/results/fig2_stance_boxplot.pdf", bbox_inches="tight")
plt.show()
print("Figure 2 saved.")

In [ ]:
# ── FIGURE 3: Embedding drift EN→PA vs EN→HI by model ────────────────────────

embed_model_lang = edf.groupby(["model", "lang_pair"])["drift"].mean().reset_index()
embed_pivot = embed_model_lang[
    embed_model_lang["lang_pair"].isin(["en→hi", "en→pa"])
].pivot(index="model", columns="lang_pair", values="drift")

fig, ax = plt.subplots(figsize=(7, 4))
embed_pivot.plot(kind="bar", ax=ax, color=["steelblue", "darkorange"], edgecolor="black")
ax.set_title("Semantic Embedding Drift by Model (LaBSE)", fontsize=12)
ax.set_ylabel("Mean Cosine Drift (1 - similarity)")
ax.set_xlabel("Model")
ax.legend(title="Language Pair")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.savefig("/kaggle/working/results/fig3_embed_drift.pdf", bbox_inches="tight")
plt.show()
print("Figure 3 saved.")

In [ ]:
# ── Save final CSV for paper ──────────────────────────────────────────────────

drift_df.to_csv("/kaggle/working/results/final_table.csv", index=False)

# Print paper-ready main result table
summary = drift_df.groupby("model")[["drift_EN_HI", "drift_EN_PA"]].mean().round(2)
summary["PA_drift_larger"] = summary["drift_EN_PA"] > summary["drift_EN_HI"]
print("\n── MAIN RESULT: Mean drift by model ──")
print(summary.to_string())
print(f"\nAll results saved to /kaggle/working/results/")